# 03 - Knowledge Graph Exploration

This notebook explores the RDF knowledge graph (`crime_graph.ttl`) built from
the Interior Minister datasets. We query the graph with SPARQL, compute
structural metrics with NetworkX, and visualize subgraphs with pyvis.

**Tools:** rdflib for RDF parsing and SPARQL, NetworkX for graph analytics, pyvis for interactive visualization.

In [ ]:
!pip install -q rdflib networkx pyvis

In [ ]:
from rdflib import Graph, Namespace, RDF, RDFS
from rdflib.plugins.sparql import prepareQuery
import networkx as nx
from pyvis.network import Network
from collections import Counter
from pathlib import Path
import pandas as pd

In [ ]:
# --- Download crime_graph.ttl from GCS ---
!gcloud auth login

GCS_BUCKET = "gs://interior-minister-data/ontology"
DATA_DIR = Path("/content/data/ontology")
DATA_DIR.mkdir(parents=True, exist_ok=True)

!gsutil cp {GCS_BUCKET}/crime_graph.ttl {DATA_DIR}/crime_graph.ttl

TTL_PATH = DATA_DIR / "crime_graph.ttl"
print(f"File size: {TTL_PATH.stat().st_size / 1024:.1f} KB")

In [ ]:
# --- Load RDF graph and show basic stats ---
g = Graph()
g.parse(str(TTL_PATH), format="turtle")

print(f"Total triples: {len(g):,}")

# Count distinct subjects, predicates, objects
subjects = set(g.subjects())
predicates = set(g.predicates())
objects = set(g.objects())

print(f"Distinct subjects: {len(subjects):,}")
print(f"Distinct predicates: {len(predicates):,}")
print(f"Distinct objects: {len(objects):,}")

# List all namespaces
print(f"\nNamespaces:")
for prefix, uri in g.namespaces():
    print(f"  {prefix}: {uri}")

# Count triples per predicate
pred_counts = Counter(str(p) for p in g.predicates())
print(f"\nTop 15 predicates by usage:")
for pred, count in pred_counts.most_common(15):
    short = pred.split("/")[-1].split("#")[-1]
    print(f"  {short}: {count:,}")

In [ ]:
# --- SPARQL: Count incidents by crime type ---
# Adjust the namespace URI and property names to match your ontology
QUERY_CRIME_TYPES = """
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX crime: <http://interior-minister.uy/ontology#>

SELECT ?crimeType (COUNT(?incident) AS ?count)
WHERE {
    ?incident rdf:type crime:Incident .
    ?incident crime:hasCrimeType ?crimeType .
}
GROUP BY ?crimeType
ORDER BY DESC(?count)
LIMIT 20
"""

results = g.query(QUERY_CRIME_TYPES)

crime_type_data = []
for row in results:
    label = str(row.crimeType).split("/")[-1].split("#")[-1]
    crime_type_data.append({"crime_type": label, "count": int(row.count)})

df_crimes = pd.DataFrame(crime_type_data)
print("Incidents by Crime Type (from SPARQL):")
display(df_crimes)

In [ ]:
# --- SPARQL: Department with most incidents ---
QUERY_DEPARTMENTS = """
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX crime: <http://interior-minister.uy/ontology#>

SELECT ?department (COUNT(?incident) AS ?count)
WHERE {
    ?incident rdf:type crime:Incident .
    ?incident crime:inDepartment ?department .
}
GROUP BY ?department
ORDER BY DESC(?count)
"""

results = g.query(QUERY_DEPARTMENTS)

dept_data = []
for row in results:
    label = str(row.department).split("/")[-1].split("#")[-1]
    dept_data.append({"department": label, "count": int(row.count)})

df_depts = pd.DataFrame(dept_data)
print("Incidents by Department (from SPARQL):")
display(df_depts)

if not df_depts.empty:
    top = df_depts.iloc[0]
    print(f"\nDepartment with most incidents: {top['department']} ({top['count']:,})")

In [ ]:
# --- Convert RDF graph to NetworkX for structural analysis ---
# Build a directed graph from all triples
nx_graph = nx.DiGraph()

for s, p, o in g:
    s_label = str(s).split("/")[-1].split("#")[-1][:50]
    o_label = str(o).split("/")[-1].split("#")[-1][:50]
    p_label = str(p).split("/")[-1].split("#")[-1][:30]
    nx_graph.add_edge(s_label, o_label, label=p_label)

print(f"NetworkX graph:")
print(f"  Nodes: {nx_graph.number_of_nodes():,}")
print(f"  Edges: {nx_graph.number_of_edges():,}")
print(f"  Density: {nx.density(nx_graph):.6f}")

# Degree distribution
in_degrees = dict(nx_graph.in_degree())
out_degrees = dict(nx_graph.out_degree())

top_in = sorted(in_degrees.items(), key=lambda x: x[1], reverse=True)[:10]
top_out = sorted(out_degrees.items(), key=lambda x: x[1], reverse=True)[:10]

print(f"\nTop 10 nodes by in-degree (most referenced):")
for node, deg in top_in:
    print(f"  {node}: {deg}")

print(f"\nTop 10 nodes by out-degree (most connections):")
for node, deg in top_out:
    print(f"  {node}: {deg}")

# Betweenness centrality on a sample (full graph may be too large)
if nx_graph.number_of_nodes() <= 5000:
    centrality = nx.betweenness_centrality(nx_graph, k=min(100, nx_graph.number_of_nodes()))
    top_central = sorted(centrality.items(), key=lambda x: x[1], reverse=True)[:10]
    print(f"\nTop 10 nodes by betweenness centrality:")
    for node, cent in top_central:
        print(f"  {node}: {cent:.4f}")
else:
    print(f"\nGraph too large ({nx_graph.number_of_nodes()} nodes) for full centrality -- sampling.")
    subgraph_nodes = [n for n, d in sorted(in_degrees.items(), key=lambda x: x[1], reverse=True)[:500]]
    sub = nx_graph.subgraph(subgraph_nodes)
    centrality = nx.betweenness_centrality(sub, k=min(50, sub.number_of_nodes()))
    top_central = sorted(centrality.items(), key=lambda x: x[1], reverse=True)[:10]
    print(f"Top 10 nodes by betweenness centrality (top-500 subgraph):")
    for node, cent in top_central:
        print(f"  {node}: {cent:.4f}")

In [ ]:
# --- Visualize a subgraph with pyvis ---
# Select the top N most-connected nodes for a readable visualization
TOP_N = 80

total_degree = {n: in_degrees.get(n, 0) + out_degrees.get(n, 0) for n in nx_graph.nodes()}
top_nodes = [n for n, _ in sorted(total_degree.items(), key=lambda x: x[1], reverse=True)[:TOP_N]]
sub = nx_graph.subgraph(top_nodes)

print(f"Subgraph for visualization: {sub.number_of_nodes()} nodes, {sub.number_of_edges()} edges")

# Create pyvis network
net = Network(
    height="700px",
    width="100%",
    directed=True,
    notebook=True,
    cdn_resources="in_line",
)

# Color nodes by type heuristic
for node in sub.nodes():
    deg = total_degree.get(node, 1)
    size = min(10 + deg * 2, 50)

    # Simple type heuristic based on node label patterns
    if any(kw in node.lower() for kw in ["department", "montevideo", "canelones"]):
        color = "#e74c3c"  # red for locations
    elif any(kw in node.lower() for kw in ["incident", "crime", "delito"]):
        color = "#3498db"  # blue for incidents
    elif any(kw in node.lower() for kw in ["type", "tipo", "category"]):
        color = "#2ecc71"  # green for categories
    else:
        color = "#95a5a6"  # grey for other

    net.add_node(node, label=node, size=size, color=color, title=f"{node} (degree: {deg})")

for u, v, data in sub.edges(data=True):
    label = data.get("label", "")
    net.add_edge(u, v, title=label, label=label, font={"size": 8})

# Physics settings for readability
net.set_options("""
{
  "physics": {
    "forceAtlas2Based": {
      "gravitationalConstant": -50,
      "centralGravity": 0.01,
      "springLength": 100,
      "springConstant": 0.08
    },
    "solver": "forceAtlas2Based",
    "stabilization": {"iterations": 150}
  },
  "edges": {
    "arrows": {"to": {"enabled": true, "scaleFactor": 0.5}},
    "smooth": {"type": "continuous"}
  }
}
""")

# Save and display
output_path = "/content/knowledge_graph.html"
net.save_graph(output_path)
print(f"Graph saved to {output_path}")

# Display in Colab
from IPython.display import HTML
HTML(filename=output_path)

## Summary

### Knowledge Graph Structure

1. **Graph Scale:** The RDF graph encodes relationships between crime incidents,
   geographic locations (departments, seccionales), crime types, and time periods
   as linked data.

2. **SPARQL Insights:** Querying by crime type and department confirms the
   patterns observed in the tabular EDA -- dominant crime categories and
   geographic hotspots are consistent across representations.

3. **Graph Metrics:** Degree distribution reveals hub nodes (departments,
   common crime types) that connect large portions of the graph. Betweenness
   centrality highlights bridge entities linking different data domains.

4. **Visual Exploration:** The pyvis subgraph visualization makes the
   entity-relationship structure tangible, showing how incidents connect to
   locations, types, and temporal dimensions.

### Potential Applications

- **Linked Data Queries:** Federate with external datasets (census, education)
  via standard SPARQL endpoints.
- **Pattern Detection:** Use graph algorithms (community detection, path finding)
  to discover non-obvious crime patterns.
- **Semantic Search:** Enable natural language queries over the knowledge graph
  for policy analysts and researchers.